In [ ]:
!pip install linearmodels -q

import pandas as pd
import numpy as np
from linearmodels import PanelOLS
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
MASTER_URL = "https://raw.githubusercontent.com/darrentweng/wharton-uss-academic-journal-executive-sentiment/refs/heads/main/data/master_dataset.csv"

df_raw = pd.read_csv(MASTER_URL)
df_raw['date'] = pd.to_datetime(df_raw['date'])
print(f"Master dataset loaded: {len(df_raw)} rows, {df_raw['ticker'].nunique()} tickers")
print(f"Columns: {list(df_raw.columns)}")
print(f"Date range: {df_raw['date'].min().date()} to {df_raw['date'].max().date()}")


In [ ]:
df_raw["log_vol_post"] = np.log(df_raw["volatility_post_10d"])
df_raw["log_vol_pre"]  = np.log(df_raw["volatility_pre_10d"])
df_raw["quarter"] = df_raw["date"].dt.to_period("Q").apply(lambda x: x.ordinal)

In [ ]:
CORE_COLS = ["log_vol_post", "log_vol_pre", "sentiment_score",
             "uncertainty_score", "surprise_factor", "ticker", "quarter"]

def prep(data, extra_cols=None):
    """Drop nulls on required columns and set panel index."""
    cols = CORE_COLS + (extra_cols or [])
    return (data.dropna(subset=cols)
                .set_index(["ticker", "quarter"]))

def run_fe(dep, exog_cols, data):
    """Run two-way fixed effects with clustered standard errors."""
    return PanelOLS(
        dependent=data[dep],
        exog=data[exog_cols],
        entity_effects=True,
        time_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)

def coef_table(model):
    """Extract clean coefficient table."""
    return pd.DataFrame({
        "coefficient": model.params,
        "std_error":   model.std_errors,
        "t_stat":      model.tstats,
        "p_value":     model.pvalues,
        "ci_lower":    model.conf_int()["lower"],
        "ci_upper":    model.conf_int()["upper"],
    }).round(6)

def print_results(label, model):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(model.summary)
    ct = coef_table(model)
    print(ct.to_string())
    print(f"\nR² (within): {model.rsquared_within:.4f}")
    print(f"Observations: {model.nobs}")
    return ct

# ── 4. Baseline OLS (Raw Volatility, No Log) ─────────────────
print("\n" + "="*60)
print("  BASELINE OLS 1 — Raw volatility, no surprise_factor")
print("="*60)

df_ols = df_raw.dropna(subset=["volatility_post_10d", "volatility_pre_10d",
                                "sentiment_score", "uncertainty_score"])
m_ols1 = smf.ols(
    "volatility_post_10d ~ uncertainty_score + sentiment_score + volatility_pre_10d",
    data=df_ols
).fit()
print(m_ols1.summary())
print(f"Observations: {int(m_ols1.nobs)}")

print("\n" + "="*60)
print("  BASELINE OLS 2 — Raw volatility, with surprise_factor")
print("="*60)

df_ols2 = df_raw.dropna(subset=["volatility_post_10d", "volatility_pre_10d",
                                  "sentiment_score", "uncertainty_score", "surprise_factor"])
m_ols2 = smf.ols(
    "volatility_post_10d ~ uncertainty_score + sentiment_score + volatility_pre_10d + surprise_factor",
    data=df_ols2
).fit()
print(m_ols2.summary())
print(f"Observations: {int(m_ols2.nobs)}")


In [ ]:
print("\n" + "="*60)
print("  BASELINE OLS 1 — Raw volatility, no surprise_factor")
print("="*60)

df_ols = df_raw.dropna(subset=["volatility_post_10d", "volatility_pre_10d",
                                "sentiment_score", "uncertainty_score"])
m_ols1 = smf.ols(
    "volatility_post_10d ~ uncertainty_score + sentiment_score + volatility_pre_10d",
    data=df_ols
).fit()
print(m_ols1.summary())
print(f"Observations: {int(m_ols1.nobs)}")

print("\n" + "="*60)
print("  BASELINE OLS 2 — Raw volatility, with surprise_factor")
print("="*60)

df_ols2 = df_raw.dropna(subset=["volatility_post_10d", "volatility_pre_10d",
                                  "sentiment_score", "uncertainty_score", "surprise_factor"])
m_ols2 = smf.ols(
    "volatility_post_10d ~ uncertainty_score + sentiment_score + volatility_pre_10d + surprise_factor",
    data=df_ols2
).fit()
print(m_ols2.summary())
print(f"Observations: {int(m_ols2.nobs)}")

In [ ]:
df_fe = prep(df_raw)

ct_full = print_results(
    "PRIMARY MODEL — Two-Way Fixed Effects (Full)",
    run_fe("log_vol_post",
           ["uncertainty_score", "sentiment_score", "log_vol_pre", "surprise_factor"],
           df_fe)
)
ct_full.to_csv("/content/fe_primary.csv")
print("Saved: fe_primary.csv")

In [ ]:
m_sentiment_only = run_fe(
    "log_vol_post",
    ["sentiment_score", "log_vol_pre", "surprise_factor"],
    df_fe
)
ct_sent = print_results("H2 TEST — Sentiment Only", m_sentiment_only)

m_full = run_fe(
    "log_vol_post",
    ["uncertainty_score", "sentiment_score", "log_vol_pre", "surprise_factor"],
    df_fe
)

print(f"\n=== H2 Comparison ===")
print(f"Full model R² (within):         {m_full.rsquared_within:.4f}")
print(f"Sentiment-only R² (within):     {m_sentiment_only.rsquared_within:.4f}")
print(f"R² gain from uncertainty_score: {m_full.rsquared_within - m_sentiment_only.rsquared_within:.4f}")

In [ ]:
df_qa = prep(df_raw, extra_cols=["sentiment_score_qa", "uncertainty_score_qa"])

ct_qa = print_results(
    "Q&A SUBSTITUTION — Replace prepared remarks with Q&A scores",
    run_fe("log_vol_post",
           ["sentiment_score_qa", "uncertainty_score_qa", "log_vol_pre", "surprise_factor"],
           df_qa)
)
ct_qa.to_csv("/content/fe_qa_substitution.csv")
print("Saved: fe_qa_substitution.csv")

In [ ]:
df_combined = prep(df_raw, extra_cols=["sentiment_score_qa", "uncertainty_score_qa"])

ct_combined = print_results(
    "COMBINED MODEL — All four linguistic predictors",
    run_fe("log_vol_post",
           ["uncertainty_score", "sentiment_score",
            "uncertainty_score_qa", "sentiment_score_qa",
            "log_vol_pre", "surprise_factor"],
           df_combined)
)
ct_combined.to_csv("/content/fe_combined.csv")
print("Saved: fe_combined.csv")

In [ ]:
for window in [5, 20]:
    post_col = f"volatility_post_{window}d"
    pre_col  = f"volatility_pre_{window}d"

    if post_col not in df_raw.columns:
        print(f"\nSkipping {window}d robustness — column {post_col} not in dataset")
        continue

    df_raw[f"log_vol_post_{window}d"] = np.log(df_raw[post_col])
    df_raw[f"log_vol_pre_{window}d"]  = np.log(df_raw[pre_col])

    log_post = f"log_vol_post_{window}d"
    log_pre  = f"log_vol_pre_{window}d"

    df_w = (df_raw.dropna(subset=[log_post, log_pre, "sentiment_score",
                                   "uncertainty_score", "surprise_factor",
                                   "ticker", "quarter"])
                  .set_index(["ticker", "quarter"]))

    ct_w = print_results(
        f"ROBUSTNESS — {window}-day volatility window",
        run_fe(log_post,
               ["uncertainty_score", "sentiment_score", log_pre, "surprise_factor"],
               df_w)
    )
    ct_w.to_csv(f"/content/fe_robustness_{window}d.csv")
    print(f"Saved: fe_robustness_{window}d.csv")

In [ ]:
df_raw["realized_variance"] = df_raw["volatility_post_10d"] ** 2 * 9
df_raw["log_realized_var"]  = np.log(df_raw["realized_variance"])

df_rv = (df_raw.dropna(subset=["log_realized_var", "log_vol_pre", "sentiment_score",
                                 "uncertainty_score", "surprise_factor",
                                 "ticker", "quarter"])
               .set_index(["ticker", "quarter"]))

ct_rv = print_results(
    "ROBUSTNESS — Realized Variance as outcome",
    run_fe("log_realized_var",
           ["uncertainty_score", "sentiment_score", "log_vol_pre", "surprise_factor"],
           df_rv)
)
ct_rv.to_csv("/content/fe_realized_variance.csv")
print("Saved: fe_realized_variance.csv")

In [ ]:
for threshold in [6, 12]:
    counts = df_raw.groupby("ticker").size()
    tickers_t = counts[counts >= threshold].index
    df_t = df_raw[df_raw["ticker"].isin(tickers_t)]
    df_t = (df_t.dropna(subset=CORE_COLS)
                .set_index(["ticker", "quarter"]))

    ct_t = print_results(
        f"ROBUSTNESS — Coverage threshold ≥{threshold} transcripts",
        run_fe("log_vol_post",
               ["uncertainty_score", "sentiment_score", "log_vol_pre", "surprise_factor"],
               df_t)
    )
    ct_t.to_csv(f"/content/fe_robustness_threshold_{threshold}.csv")
    print(f"Saved: fe_robustness_threshold_{threshold}.csv")

In [ ]:
df_eps = df_raw[df_raw["eps_surprise"].notna()].copy()
df_eps["surprise_factor"] = df_eps["eps_surprise"]

df_eps_fe = (df_eps.dropna(subset=CORE_COLS)
                   .set_index(["ticker", "quarter"]))

ct_eps = print_results(
    "EPS SURPRISE ONLY — Clean subset robustness check",
    run_fe("log_vol_post",
           ["uncertainty_score", "sentiment_score", "log_vol_pre", "surprise_factor"],
           df_eps_fe)
)
ct_eps.to_csv("/content/fe_eps_only.csv")
print("Saved: fe_eps_only.csv")
print(f"Tickers in EPS subset: {df_eps['ticker'].nunique()}")

In [ ]:
df_granger = df_raw.sort_values(["ticker", "date"]).copy()
df_granger["uncertainty_lag1"] = df_granger.groupby("ticker")["uncertainty_score"].shift(1)

df_g = (df_granger.dropna(subset=["log_vol_post", "log_vol_pre",
                                    "uncertainty_lag1", "ticker", "quarter"])
                  .set_index(["ticker", "quarter"]))

m_restricted   = run_fe("log_vol_post", ["log_vol_pre"], df_g)
m_unrestricted = run_fe("log_vol_post", ["log_vol_pre", "uncertainty_lag1"], df_g)

ct_granger = print_results("GRANGER CAUSALITY — Lagged uncertainty", m_unrestricted)
ct_granger.to_csv("/content/fe_granger.csv")

granger_summary = pd.DataFrame([{
    "restricted_r2":         round(m_restricted.rsquared_within, 6),
    "unrestricted_r2":       round(m_unrestricted.rsquared_within, 6),
    "r2_gain":               round(m_unrestricted.rsquared_within - m_restricted.rsquared_within, 6),
    "uncertainty_lag1_coef": round(m_unrestricted.params["uncertainty_lag1"], 6),
    "uncertainty_lag1_pval": round(m_unrestricted.pvalues["uncertainty_lag1"], 6),
}])
granger_summary.to_csv("/content/fe_granger_summary.csv", index=False)
print("Saved: fe_granger.csv, fe_granger_summary.csv")

print("\n" + "="*60)
print("  ALL REGRESSIONS COMPLETE")
print("="*60)
print("\nOutput files saved to /content/:")
print("  fe_primary.csv")
print("  fe_qa_substitution.csv")
print("  fe_combined.csv")
print("  fe_robustness_5d.csv  (if 5d columns present)")
print("  fe_robustness_20d.csv (if 20d columns present)")
print("  fe_realized_variance.csv")
print("  fe_robustness_threshold_6.csv")
print("  fe_robustness_threshold_12.csv")
print("  fe_eps_only.csv")
print("  fe_granger.csv")
print("  fe_granger_summary.csv")